In [0]:
data = [
    (101, "Laptop", 70000, 1, "2026-03-01"),
    (102, "Phone", 25000, 2, "2026-03-02"),
    (103, "Laptop", 70000, 1, "2026-03-02"),
    (104, "Tablet", 30000, 1, "2026-03-03"),
    (105, "Phone", 25000, 1, "2026-03-03")
]

columns = ["order_id","product","price","quantity","order_date"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df.write \
.option("header","true") \
.mode("overwrite") \
.csv("/Volumes/workspace/default/demo_volume/sample_sales")

In [0]:
df = spark.read \
.option("header","true") \
.csv("/Volumes/workspace/default/demo_volume/sample_sales")

display(df)

BRONZE LAYER 

In [0]:
bronze_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/Volumes/workspace/default/demo_volume/sample_sales")

bronze_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_sales")

In [0]:
%sql
SELECT * FROM bronze_sales

SILVER LAYER 

In [0]:
silver_df = spark.table("bronze_sales") \
.dropDuplicates()

silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_sales")

In [0]:
%sql
SELECT * FROM silver_sales

GOLD LAYER

In [0]:
from pyspark.sql.functions import col, sum

gold_df = spark.table("silver_sales") \
.withColumn("revenue", col("price") * col("quantity")) \
.groupBy("product") \
.agg(sum("revenue").alias("total_revenue"))

In [0]:
gold_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_sales_summary")

In [0]:
%sql
SELECT * FROM gold_sales_summary
ORDER BY total_revenue DESC